In [1]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 68.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("bookcorpus/bookcorpus", revision="refs/convert/parquet")

plain_text/train/0000.parquet:   0%|          | 0.00/312M [00:00<?, ?B/s]

In [ ]:
"""import requests
from tqdm import tqdm

url = "https://huggingface.co/datasets/bookcorpus/bookcorpus/resolve/refs%2Fconvert%2Fparquet/plain_text/train/0000.parquet"
local_path = "shard_0000.parquet"

with requests.get(url, stream=True, timeout=60) as r:
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(local_path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
        for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

print("Download complete:", local_path)


import pandas as pd
from datasets import Dataset

df = pd.read_parquet(local_path)
df = df.iloc[:40000]

dataset = Dataset.from_pandas(df, preserve_index=False)
print(dataset)
print(dataset[0]["text"])"""

In [ ]:
"""import pandas as pd
from datasets import Dataset

df = pd.read_parquet(local_path)
df = df.iloc[:40000]

dataset = {"train": Dataset.from_pandas(df, preserve_index=False)}

print(dataset['train'][0]['text'])"""

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import spacy

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

nlp = spacy.load("en_core_web_sm")

def spacy_tokenizer(text):
    doc = nlp(text)

    tokens = [token.text for token in doc]

    return tokens


tokenizer = Tokenizer(BPE())

trainer = BpeTrainer(
    vocab_size=8000,
    special_tokens=[
        "<unk>",
        "<pad>",
        "<bos>",
        "<eos>",
    ],
)

tokenizer.pre_tokenizer = Whitespace()


with open("bookcorpus.txt", "w", encoding="utf-8") as f:
    for example in dataset["train"]:
        f.write(example["text"] + "\n")


tokenizer.train(
    files=["bookcorpus.txt"],
    trainer=trainer,
)

In [ ]:
print(dataset)

In [ ]:
from torch.utils.data import Dataset
import torch

class BookCorpus(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset["train"]
        self.pad_len = tokenizer.token_to_id("<pad>")

    def __len__(self):
        return len(self.dataset)

    #missed adding _pad function in dataset
    def _pad(self, ids, pad_id):
        ids = ids[:512]
        ids = ids + [pad_id] * (512 - len(ids))
        return ids

    def __getitem__(self, idx):
        text = self.dataset[idx]["text"]

        tokens = spacy_tokenizer(text)
        encoding = tokenizer.encode(" ".join(tokens))

        ids = encoding.ids

        ids = self._pad(ids,self.pad_len)

        input_ids = torch.tensor(ids[:-1], dtype=torch.long)
        target_ids = torch.tensor(ids[1:], dtype=torch.long)

        return input_ids, target_ids

In [ ]:
import math
import torch.nn as nn

class GPTEmbedding(nn.Module):
    def __init__(self, vocab_size, max_len=512, d_model=768):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)

    def forward(self, input_ids):
        seq_len = input_ids.size(1)

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        ).unsqueeze(0)

        token_embeddings = self.token_embedding(input_ids)
        position_embeddings = self.position_embedding(positions)

        return token_embeddings + position_embeddings

In [ ]:
import torch.nn as nn

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer = nn.Sequential(
        nn.Linear(768,3072),
        nn.GELU(),
        nn.Dropout(0.1),
        nn.Linear(3072,768),
    )
  def forward(self,x):
    return self.layer(x)

In [ ]:
import torch.nn as nn

class Transfomer(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()

    self.multi_attention = nn.MultiheadAttention(embed_dim=d_model,num_heads=num_heads,batch_first=True)

    self.norm1 = nn.LayerNorm(normalized_shape=d_model)

    self.feed_forward = FeedForward()

    self.norm2 = nn.LayerNorm(normalized_shape=d_model)

  def forward(self,x,attn_mask):
    attn_out, _ = self.multi_attention(x,x,x,attn_mask=attn_mask)

    x1 = self.norm1(x + attn_out)

    ff_out = self.feed_forward(x1)

    x2 = self.norm2(x1 + ff_out)

    return x2


In [ ]:
"""
seq_len = x.size(1)

causal_mask = torch.triu(
    torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool), diagonal=1
)
"""

In [ ]:
import torch.nn as nn
import torch

class GPT1(nn.Module):
  def __init__(self,vocab_size,d_model,num_heads):
    super().__init__()

    self.embedding = GPTEmbedding(
            vocab_size=vocab_size,
            max_len=512,
            d_model=d_model
        )

    self.blocks = nn.ModuleList([
                    Transfomer(
                        d_model=d_model,
                        num_heads=num_heads
                    )
                    for _ in range(12)
                ])

    self.lm_head = nn.Linear(d_model, vocab_size)

  def forward(self,x,attn_mask):
    x = self.embedding(x)
    
    for block in self.blocks:
      x = block(x, attn_mask)

    logits = self.lm_head(x)

    return logits

In [ ]:
from torch.utils.data import DataLoader,random_split

book_corpus_dataset = BookCorpus(dataset)

length = len(book_corpus_dataset)

#missed int() for the length its taking 80 as 80.0
train_len = int(0.8 * length)
val_len = length - train_len

train_dataset, val_dataset = random_split(
                                book_corpus_dataset,
                                [train_len, val_len]
                            )
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)

model = GPT1(vocab_size=tokenizer.get_vocab_size(),d_model=768,num_heads=4).to(device)

In [ ]:
import torch.optim as optim

def make_mask(x):
  seq_len = x.size(1)
  attn_mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool),diagonal=1)
  return attn_mask

loss_fn = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(),lr=2.5e-4,weight_decay=0.01)

epochs = 20

for epoch in range(1, epochs + 1):
  model.train()
  train_loss = 0.0
  for x,y in train_dataloader:
    x,y = x.to(device),y.to(device)
      
    attn_mask = make_mask(x)

    optimizer.zero_grad()
    logits = model(x,attn_mask)
    loss = loss_fn(logits.reshape(-1, logits.size(-1)),y.reshape(-1))
    loss.backward()
    optimizer.step()
    train_loss += loss.item()

  model.eval()
  val_loss = 0.0
  with torch.no_grad():
    for x,y in val_dataloader:
      x,y = x.to(device),y.to(device)

      attn_mask = make_mask(x)

      logits = model(x,attn_mask)

      loss = loss_fn(logits.reshape(-1, logits.size(-1)),y.reshape(-1))

      val_loss += loss.item()

  print(f"Epoch {epoch}, Train_loss {train_loss/len(train_dataloader)} , Val_loss {val_loss/len(val_dataloader)}")

In [ ]:

@torch.inference_mode()
def inferencing(txt,max_tokens,model):
  model.eval()

  txt = spacy_tokenizer(txt)

  encoding = tokenizer.encode(" ".join(txt))

  input_ids = torch.tensor([encoding.ids], dtype=torch.long, device=device)

  for _ in range(max_tokens):
    attn_mask = make_mask(input_ids)

    logits = model(input_ids,attn_mask)

    next_token = logits[:, -1].argmax(dim=-1)

    input_ids = torch.cat(
        [input_ids, next_token.unsqueeze(1)],
        dim=1
    )
  return tokenizer.decode(input_ids[0].tolist())

prompt = "name famous books"

print(inferencing(prompt,max_tokens=50, model=model))